# Random Forest model
Train a fresh Random Forest with identical settings on each chronological split and predict the probability of illicit activity (`label = 1`). Use only the 165 supplied features and known labels.

The split boundaries live in `validation_splits.json`, also read by notebook 1. No separate CSV copies of validation data are needed. This notebook runs independently of the inspection notebooks.

Both periods are evaluated here. If you use both scores to select models or settings, both are development validation folds: the name “latest holdout” does **not** make its score an untouched final evaluation.

Required packages: pandas, numpy, scikit-learn, and a Python Jupyter kernel.

In [1]:
from pathlib import Path
import json
import time
import platform
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "bt-4012-competition-2026").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the competition project.")
DATA = ROOT / "bt-4012-competition-2026"
splits = json.loads((ROOT / "validation_splits.json").read_text())
FEATURES = [f"feat_{i}" for i in range(1, 166)]
SEED = 42
THRESHOLD = 0.5
MODEL_PARAMS = dict(
    n_estimators=200, max_depth=None, min_samples_leaf=1,
    max_features="sqrt", class_weight=None, random_state=SEED, n_jobs=2,
)
print("Python:", platform.python_version(), "scikit-learn:", sklearn.__version__)
display(pd.DataFrame(splits))

Python: 3.13.1 scikit-learn: 1.7.1


,name,train_start,train_end,validation_start,validation_end
0,earlier check,1,21,22,28
1,latest holdout,1,28,29,35


## Load known labels and check inputs
Unknown labels cannot supervise a classifier. No feature imputation is needed if the completeness check passes. Trees do not require feature scaling. `txId` is retained for overlap checks; it is excluded from the predictors, along with `time_step`.

The initial Random Forest model uses no class weighting or resampling. These can be separate experiments later.

In [2]:
data = pd.read_csv(DATA / "train.csv", usecols=["txId", "time_step", *FEATURES, "label"])
unexpected = data.loc[data.label.notna() & ~data.label.isin([0, 1]), "label"].unique()
if len(unexpected):
    raise ValueError(f"Unexpected labels: {unexpected}")
labelled = data.loc[data.label.isin([0, 1])].copy()
labelled["label"] = labelled["label"].astype(int)
assert labelled.txId.is_unique, "Duplicate transaction IDs need investigation."
assert labelled.time_step.notna().all(), "Missing time steps need investigation."
assert np.isfinite(labelled[FEATURES].to_numpy()).all(), "Missing/infinite features need investigation."
print(f"Known labels available: {len(labelled):,}")
print(f"Unknown labels excluded: {data.label.isna().sum():,}")
print(f"Predictor columns: {len(FEATURES)}")
display(labelled.label.value_counts().rename_axis("label").to_frame("count"))

Known labels available: 31,235
Unknown labels excluded: 110,537
Predictor columns: 165


,count
label,
0,27591
1,3644


## Fit and evaluate both chronological folds
Each loop iteration creates a new model. ROC-AUC and average precision use probabilities, without a classification threshold. Precision, recall, F1, and the confusion matrix use a fixed threshold of 0.5; this threshold has not been tuned.

Average precision summarizes the precision–recall curve (it is not the trapezoidal PR-AUC). The training ROC-AUC is a diagnostic for memorization, not a generalization score. No validation labels or validation-fitted transformations enter training.

In [3]:
results = []
confusions = {}
for split in splits:
    train_fold = labelled.loc[labelled.time_step.between(split["train_start"], split["train_end"])]
    val_fold = labelled.loc[labelled.time_step.between(split["validation_start"], split["validation_end"])]
    assert split["train_end"] < split["validation_start"], "Time windows overlap."
    assert not set(train_fold.txId) & set(val_fold.txId), "Transaction overlap detected."
    assert train_fold.label.nunique() == val_fold.label.nunique() == 2, "Both classes are required."
    X_train, y_train = train_fold[FEATURES], train_fold.label
    X_val, y_val = val_fold[FEATURES], val_fold.label

    model = RandomForestClassifier(**MODEL_PARAMS)
    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started
    illicit_column = list(model.classes_).index(1)
    probabilities = model.predict_proba(X_val)[:, illicit_column]
    train_probabilities = model.predict_proba(X_train)[:, illicit_column]
    predictions = (probabilities >= THRESHOLD).astype(int)

    results.append({
        "split": split["name"], "train_steps": f'{split["train_start"]}–{split["train_end"]}',
        "validation_steps": f'{split["validation_start"]}–{split["validation_end"]}',
        "train_rows": len(train_fold), "validation_rows": len(val_fold),
        "validation_illicit": int(y_val.sum()), "validation_illicit_share": float(y_val.mean()),
        "train_roc_auc": roc_auc_score(y_train, train_probabilities),
        "validation_roc_auc": roc_auc_score(y_val, probabilities),
        "average_precision": average_precision_score(y_val, probabilities),
        "threshold": THRESHOLD,
        "illicit_precision": precision_score(y_val, predictions, zero_division=0),
        "illicit_recall": recall_score(y_val, predictions, zero_division=0),
        "illicit_f1": f1_score(y_val, predictions, zero_division=0),
        "fit_seconds": fit_seconds,
    })
    confusions[split["name"]] = pd.DataFrame(
        confusion_matrix(y_val, predictions, labels=[0, 1]),
        index=["Actual licit", "Actual illicit"],
        columns=["Predicted licit", "Predicted illicit"],
    )
    print(f'{split["name"]}: ROC-AUC = {results[-1]["validation_roc_auc"]:.4f}; '
          f'fit time = {fit_seconds:.1f}s', flush=True)

results_df = pd.DataFrame(results)
with pd.option_context("display.max_columns", None):
    display(results_df.round(4))
for name, matrix in confusions.items():
    print(name)
    display(matrix)

earlier check: ROC-AUC = 0.8765; fit time = 4.2s


latest holdout: ROC-AUC = 0.9950; fit time = 5.4s


,split,train_steps,validation_steps,train_rows,validation_rows,validation_illicit,validation_illicit_share,train_roc_auc,validation_roc_auc,average_precision,threshold,illicit_precision,illicit_recall,illicit_f1,fit_seconds
0,earlier check,1–21,22–28,19530,5677,671,0.1182,1.0,0.8765,0.8469,0.5,0.9754,0.7675,0.8590,4.1709
1,latest holdout,1–28,29–35,25207,6028,1102,0.1828,1.0,0.9950,0.9855,0.5,0.9952,0.9319,0.9625,5.4341


earlier check


,Predicted licit,Predicted illicit
Actual licit,4993,13
Actual illicit,156,515


latest holdout


,Predicted licit,Predicted illicit
Actual licit,4921,5
Actual illicit,75,1027


## Compare folds and save the experiment
Inspect each fold's result before the mean: performance may differ across periods. The mean below weights the two folds equally. These two scores are not a confidence interval.

The saved summary and settings allow later feature or model experiments to use the same reference. Rerunning this notebook replaces this model's result files.

In [4]:
metrics = ["validation_roc_auc", "average_precision", "illicit_precision", "illicit_recall", "illicit_f1"]
display(results_df.set_index("split")[metrics].round(4))
print(f'Mean fold ROC-AUC: {results_df.validation_roc_auc.mean():.4f}')
print(f'Fold ROC-AUC range: {results_df.validation_roc_auc.min():.4f}–{results_df.validation_roc_auc.max():.4f}')

RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
results_df.to_csv(RESULTS / "random_forest_model.csv", index=False)
settings = {"model": "RandomForestClassifier", "parameters": MODEL_PARAMS,
            "features": FEATURES, "threshold": THRESHOLD, "splits": splits,
            "python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit_learn": sklearn.__version__}
(RESULTS / "random_forest_model_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print("Saved metrics and settings in results/.")

,validation_roc_auc,average_precision,illicit_precision,illicit_recall,illicit_f1
split,,,,,
earlier check,0.8765,0.8469,0.9754,0.7675,0.8590
latest holdout,0.9950,0.9855,0.9952,0.9319,0.9625


Mean fold ROC-AUC: 0.9357
Fold ROC-AUC range: 0.8765–0.9950
Saved metrics and settings in results/.


## Train the final Random Forest model and create a Kaggle submission
After validation, fit a **new model on all known labels** in steps 1–35 using the same Random Forest settings. Previously held-out training rows can now be used: the final model predicts the separate, later test transactions. The validation scores above belong to the earlier fold models.

Submit the probability of class 1 (illicit), without applying the 0.5 threshold. Match predictions to the sample submission by `index`, preserving its row order. The file is saved under `submissions/`, which is excluded from Git. Rerunning this section replaces `random_forest_model.csv`; uploading it to Kaggle is a separate step.

In [5]:
test = pd.read_csv(DATA / "test.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

assert list(sample_submission.columns) == ["index", "target"], "Unexpected submission format."
assert test["index"].notna().all() and test["index"].is_unique, "Test index must be unique and complete."
assert sample_submission["index"].notna().all() and sample_submission["index"].is_unique
assert len(test) == len(sample_submission), "Test and submission row counts differ."
assert set(test["index"]) == set(sample_submission["index"]), "Submission IDs do not match test IDs."
assert test.txId.is_unique, "Duplicate test transaction IDs need investigation."
assert not set(data.txId) & set(test.txId), "Training and test transactions overlap."
assert test.time_step.min() > data.time_step.max(), "Test must contain later periods."
assert np.isfinite(test[FEATURES].to_numpy()).all(), "Test features contain missing/infinite values."

final_model = RandomForestClassifier(**MODEL_PARAMS)
final_model.fit(labelled[FEATURES], labelled["label"])
illicit_column = list(final_model.classes_).index(1)
test_probabilities = final_model.predict_proba(test[FEATURES])[:, illicit_column]

# Match by the submission ID instead of assuming that both files have the same row order.
probability_by_index = pd.Series(test_probabilities, index=test["index"])
submission = sample_submission.copy()
submission["target"] = submission["index"].map(probability_by_index)
assert np.isfinite(submission["target"]).all(), "Missing/non-finite predictions."
assert submission["target"].between(0, 1).all(), "Probabilities must be in [0, 1]."

SUBMISSIONS = ROOT / "submissions"
SUBMISSIONS.mkdir(exist_ok=True)
submission_path = SUBMISSIONS / "random_forest_model.csv"
submission.to_csv(submission_path, index=False)

# Read back the actual saved file to verify its headers, row order, and probabilities.
saved_submission = pd.read_csv(submission_path)
assert list(saved_submission.columns) == list(sample_submission.columns)
assert saved_submission["index"].equals(sample_submission["index"])
assert len(saved_submission) == len(test)
assert np.isfinite(saved_submission["target"]).all()
assert saved_submission["target"].between(0, 1).all()
assert np.allclose(saved_submission["target"], submission["target"], rtol=0, atol=1e-12)

print(f"Final model trained on {len(labelled):,} labelled transactions.")
print(f"Saved {len(saved_submission):,} predictions to submissions/{submission_path.name}")
print("Submission checks passed: headers, IDs, row order, count, and probability range.")
display(saved_submission.head())
display(saved_submission["target"].describe())

Final model trained on 31,235 labelled transactions.
Saved 15,329 predictions to submissions/random_forest_model.csv
Submission checks passed: headers, IDs, row order, count, and probability range.


,index,target
0,0,0.015
1,1,0.035
2,2,0.020
3,3,0.025
4,4,0.005


count    15329.000000
mean         0.082190
std          0.185156
min          0.000000
25%          0.005000
50%          0.020000
75%          0.075000
max          1.000000
Name: target, dtype: float64

## Kaggle leaderboard result

After uploading `submissions/random_forest_model.csv`, the user reported a **public ROC-AUC of 0.92447** on the visible **30%** of the test set. This is an externally observed score entered manually, not an output calculated by this notebook. The hidden private score is not yet known. The result is recorded alongside the two chronological validation scores in `results/experiment_log.csv`. Do not repeatedly select model changes only by the public leaderboard score.